In [1]:
# ============================================================================
# ENVIRONMENT SELECTION (Dataset preparation notebook)
# ============================================================================
RUN_ON_OSC = True
if RUN_ON_OSC:
    PARENT_DIR = "/fs/scratch/<allocation>"
    print("🖥️  Running on an HPC cluster")
else:
    PARENT_DIR = r"D:\Poweline\DL\Data"
    print("💻 Running locally (Windows)")
print(f"📁 Parent directory: {PARENT_DIR}")

🖥️  Running on an HPC cluster
📁 Parent directory: /fs/scratch/<allocation>


# ASR Dataset Preparation
This notebook prepares datasets for ASR training: parsing timings, transcribing MP3 with Whisper, extracting aligned audio from .bin files, building HuggingFace datasets, preprocessing (feature extraction + tokenization), creating a data collator, and saving artifacts to disk under `OUTPUT_DIR/hf_datasets`.

In [ ]:
# --------------------------------------------------------------------------
# REQUIRED CONFIG / VARIABLES
# --------------------------------------------------------------------------
import os
import json
import numpy as np
from scipy import signal
import soundfile as sf
import torch
from datasets import Dataset, Audio
import jiwer
from dataclasses import dataclass

# If the environment-selection cell hasn't run or PARENT_DIR wasn't set, provide a sensible default.
if 'PARENT_DIR' not in globals():
    PARENT_DIR = '/fs/scratch/<allocation>'  # change if your data lives elsewhere

# Paths (adjust if your data is located elsewhere)
TIMING_FILE = os.path.join(PARENT_DIR, 'alice_timing_analysis.txt')
MP3_DIR = os.path.join(PARENT_DIR, 'Alice_In_Wonderland_mp3')
BIN_DIR = os.path.join(PARENT_DIR, 'May29_Alice')
TRANSCRIPTS_DIR = os.path.join(PARENT_DIR, 'transcripts')
OUTPUT_DIR = os.path.join(PARENT_DIR, 'output')

# Audio and processing settings
TARGET_SR = 16000  # target sample rate used by ASR pipeline

# Ensure output folders exist
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device selection for model training/inference (dataset notebook doesn't load the model)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Configuration summary:')
print(f'  PARENT_DIR:      {PARENT_DIR}')
print(f'  TIMING_FILE:     {TIMING_FILE} (exists: {os.path.exists(TIMING_FILE)})')
print(f'  MP3_DIR:         {MP3_DIR} (exists: {os.path.isdir(MP3_DIR)})')
print(f'  BIN_DIR:         {BIN_DIR} (exists: {os.path.isdir(BIN_DIR)})')
print(f'  TRANSCRIPTS_DIR: {TRANSCRIPTS_DIR}')
print(f'  OUTPUT_DIR:      {OUTPUT_DIR}')
print(f'  TARGET_SR:       {TARGET_SR}')
print(f'  device:          {device}')

/users/PAS1567/user/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration summary:
  PARENT_DIR:      /fs/scratch/<allocation>
  TIMING_FILE:     /fs/scratch/<allocation>/alice_timing_analysis.txt (exists: True)
  MP3_DIR:         /fs/scratch/<allocation>/Alice_In_Wonderland_mp3 (exists: True)
  BIN_DIR:         /fs/scratch/<allocation>/May29_Alice (exists: True)
  TRANSCRIPTS_DIR: /fs/scratch/<allocation>/transcripts
  OUTPUT_DIR:      /fs/scratch/<allocation>/output
  TARGET_SR:       16000
  device:          cpu


## Step 3: Parse Timing Information

In [3]:
def parse_timing_file(timing_file):
    """
    Parse alice_timing_analysis.txt to extract chapter timing information.
    Returns: list of dicts with chapter timing data
    """
    chapters = []
    
    with open(timing_file, 'r') as f:
        content = f.read()
    
    # Parse the summary table section
    lines = content.split('\n')
    in_table = False
    
    for line in lines:
        if 'Chapter' in line and 'MP3 (s)' in line:
            in_table = True
            continue
        
        if in_table:
            if line.startswith('---'):
                continue
            if not line.strip():
                break
            
            parts = line.split()
            if len(parts) >= 5:
                try:
                    chapter_num = int(parts[0])
                    chapters.append({
                        'chapter': chapter_num,
                        'mp3_duration': float(parts[1]),
                        'bin_duration': float(parts[2]),
                        'start_offset': float(parts[3]),
                        'stop_time': float(parts[4])
                    })
                except ValueError:
                    continue
    
    return chapters

# Load timing information
timing_info = parse_timing_file(TIMING_FILE)

print(f"✅ Found {len(timing_info)} chapters with timing data:")
for info in timing_info:
    print(f"   Chapter {info['chapter']}: {info['mp3_duration']:.1f}s (offset: {info['start_offset']:.2f}s)")

✅ Found 9 chapters with timing data:
   Chapter 1: 632.0s (offset: 1.38s)
   Chapter 2: 724.1s (offset: 2.04s)
   Chapter 4: 1168.4s (offset: 1.95s)
   Chapter 5: 794.5s (offset: 2.17s)
   Chapter 6: 765.2s (offset: 2.09s)
   Chapter 7: 1027.7s (offset: 2.05s)
   Chapter 8: 789.4s (offset: 1.95s)
   Chapter 10: 1345.7s (offset: 1.63s)
   Chapter 11: 601.2s (offset: 1.61s)


## Step 4: Generate Ground Truth Transcriptions

In [ ]:
import whisper
def transcribe_mp3_with_whisper(mp3_path, chapter_num, output_dir):
    """
    Transcribe MP3 file using Whisper and save to JSON.
    Returns: dict with transcription and timing info
    """
    output_file = os.path.join(output_dir, f"chapter_{chapter_num:02d}_transcript.json")
    
    # Check if already transcribed
    if os.path.exists(output_file):
        print(f"   Loading existing transcript for Chapter {chapter_num}")
        with open(output_file, 'r') as f:
            return json.load(f)
    
    print(f"   Transcribing Chapter {chapter_num} with Whisper...")
    
    # Load Whisper model
    model = whisper.load_model("base")  # Options: tiny, base, small, medium, large
    
    # Transcribe with word-level timestamps
    result = model.transcribe(
        mp3_path,
        language="en",
        word_timestamps=True,
        verbose=False
    )
    
    # Extract segments with timing
    transcription_data = {
        'chapter': chapter_num,
        'text': result['text'],
        'segments': []
    }
    
    for segment in result['segments']:
        transcription_data['segments'].append({
            'start': segment['start'],
            'end': segment['end'],
            'text': segment['text'].strip()
        })
    
    # Save to JSON
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(transcription_data, f, indent=2, ensure_ascii=False)
    
    print(f"   ✓ Saved transcript ({len(transcription_data['segments'])} segments)")
    return transcription_data


# Generate transcriptions for all available chapters
print("Generating ground truth transcriptions...\n")

transcriptions = {}
for info in timing_info:
    chapter_num = info['chapter']
    mp3_file = os.path.join(MP3_DIR, f"Alice_In_Wonderland_ch_{chapter_num:02d}.mp3")
    
    if os.path.exists(mp3_file):
        transcript = transcribe_mp3_with_whisper(mp3_file, chapter_num, TRANSCRIPTS_DIR)
        transcriptions[chapter_num] = transcript
    else:
        print(f"   ⚠ MP3 file not found for Chapter {chapter_num}")

print(f"\n✅ Generated transcriptions for {len(transcriptions)} chapters")

In [8]:
# Load transcripts if they already exist (convenience cell)
def load_transcript_json(chapter_num, transcripts_dir):
    transcript_path = os.path.join(transcripts_dir, f"chapter_{chapter_num:02d}_transcript.json")
    if not os.path.exists(transcript_path):
        raise FileNotFoundError(transcript_path)
    with open(transcript_path, "r", encoding="utf-8") as f:
        return json.load(f)

transcriptions = {}
missing_transcripts = []
for info in timing_info:
    chapter_num = info['chapter']
    try:
        transcriptions[chapter_num] = load_transcript_json(chapter_num, TRANSCRIPTS_DIR)
        print(f"   ✓ Loaded transcript for Chapter {chapter_num}")
    except FileNotFoundError:
        missing_transcripts.append(chapter_num)
        print(f"   ⚠ Transcript file not found for Chapter {chapter_num}")
if missing_transcripts:
    missing = ', '.join(str(c) for c in missing_transcripts)
    print(f"⚠ Missing transcripts for chapters: {missing}")
else:
    print(f"✅ Loaded transcriptions for {len(transcriptions)} chapters")

   ✓ Loaded transcript for Chapter 1
   ✓ Loaded transcript for Chapter 2
   ✓ Loaded transcript for Chapter 4
   ✓ Loaded transcript for Chapter 5
   ✓ Loaded transcript for Chapter 6
   ✓ Loaded transcript for Chapter 7
   ✓ Loaded transcript for Chapter 8
   ✓ Loaded transcript for Chapter 10
   ✓ Loaded transcript for Chapter 11
✅ Loaded transcriptions for 9 chapters


## Step 5: Extract and Align Audio from .bin Files

In [9]:
def extract_aligned_audio(bin_path, start_offset, duration, usrp_sr=200_000, target_sr=16000):
    usb_data = np.fromfile(bin_path, dtype=np.float32)
    start_sample = int(start_offset * usrp_sr)
    end_sample = int((start_offset + duration) * usrp_sr)
    audio_segment = usb_data[start_sample:end_sample]
    audio_segment = audio_segment / np.max(np.abs(audio_segment) + 1e-8)
    num_samples_target = int(len(audio_segment) * target_sr / usrp_sr)
    audio_resampled = signal.resample(audio_segment, num_samples_target)
    return audio_resampled.astype(np.float32)

def create_training_dataset(timing_info, transcriptions, bin_dir, output_dir):
    dataset = []
    for info in timing_info:
        chapter_num = info['chapter']
        if chapter_num not in transcriptions:
            continue
        bin_file = os.path.join(bin_dir, f"Chap_{chapter_num}_img.bin")
        if not os.path.exists(bin_file):
            print(f"   ⚠ .bin file not found for Chapter {chapter_num}")
            continue
        print(f"   Processing Chapter {chapter_num}...")
        audio_array = extract_aligned_audio(bin_file, info['start_offset'], info['mp3_duration'])
        wav_path = os.path.join(output_dir, f"chapter_{chapter_num:02d}_aligned.wav")
        sf.write(wav_path, audio_array, TARGET_SR)
        full_text = transcriptions[chapter_num]['text'].strip()
        dataset.append({'audio': wav_path, 'text': full_text, 'chapter': chapter_num, 'duration': len(audio_array) / TARGET_SR})
    return dataset

print("Creating training dataset...")
dataset_list = create_training_dataset(timing_info, transcriptions, BIN_DIR, OUTPUT_DIR)
print(f"✅ Created dataset with {len(dataset_list)} chapters")
print(f"   Total duration: {sum([d['duration'] for d in dataset_list]) / 60:.1f} minutes")
for item in dataset_list:
    print(f"   Chapter {item['chapter']}: {item['duration']:.1f}s - {len(item['text'])} chars")

# Save the raw dataset list to disk so Step 6 (or other notebooks) can reload it without re-running extraction
DATASET_DIR = os.path.join(OUTPUT_DIR, 'hf_datasets')
os.makedirs(DATASET_DIR, exist_ok=True)
raw_dataset_path = os.path.join(DATASET_DIR, 'raw_dataset.json')
with open(raw_dataset_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_list, f, indent=2)
print(f"Saved raw dataset list to: {raw_dataset_path}")

Creating training dataset...
   Processing Chapter 1...
   Processing Chapter 2...
   Processing Chapter 2...
   Processing Chapter 4...
   Processing Chapter 4...
   Processing Chapter 5...
   Processing Chapter 5...
   Processing Chapter 6...
   Processing Chapter 6...
   Processing Chapter 7...
   Processing Chapter 7...
   Processing Chapter 8...
   Processing Chapter 8...
   Processing Chapter 10...
   Processing Chapter 10...
   Processing Chapter 11...
   Processing Chapter 11...
✅ Created dataset with 9 chapters
   Total duration: 130.8 minutes
   Chapter 1: 632.0s - 11163 chars
   Chapter 2: 724.1s - 10683 chars
   Chapter 4: 1168.4s - 13736 chars
   Chapter 5: 794.5s - 11590 chars
   Chapter 6: 765.2s - 13485 chars
   Chapter 7: 1027.7s - 12390 chars
   Chapter 8: 789.4s - 13431 chars
   Chapter 10: 1345.7s - 10802 chars
   Chapter 11: 601.2s - 10014 chars
Saved raw dataset list to: /fs/scratch/<allocation>/output/hf_datasets/raw_dataset.json
✅ Created dataset with 9 chapters

In [10]:
# --------------------------------------------------------------------------
# Load raw dataset_list from disk (created in Step 5). This is handy if you restart the kernel or want to reuse the extracted WAVs without re-running extraction)
# --------------------------------------------------------------------------
DATASET_DIR = os.path.join(OUTPUT_DIR, 'hf_datasets')
raw_dataset_path = os.path.join(DATASET_DIR, 'raw_dataset.json')
if os.path.exists(raw_dataset_path):
    with open(raw_dataset_path, 'r', encoding='utf-8') as f:
        dataset_list = json.load(f)
    print(f"Loaded dataset_list from {raw_dataset_path} ({len(dataset_list)} items)")
    total_dur = sum(item.get('duration', 0) for item in dataset_list)
    print(f"Total duration: {total_dur/60:.2f} minutes")
else:
    print(f"No raw dataset file found at {raw_dataset_path}. If you haven't run Step 5, run the extraction cell first.")

Loaded dataset_list from /fs/scratch/<allocation>/output/hf_datasets/raw_dataset.json (9 items)
Total duration: 130.80 minutes


## Step 6: Prepare Dataset for Training (split & save)

In [11]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, Audio, Value, load_from_disk
import csv

# Split dataset (80% train, 20% validation)
train_data, val_data = train_test_split(dataset_list, test_size=0.2, random_state=42)
print(f"Dataset split:")
print(f"  Training: {len(train_data)} chapters")
print(f"  Validation: {len(val_data)} chapters")

# Create HuggingFace datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

# Try casting 'audio' column to Audio for lazy loading
try:
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=TARGET_SR))
    val_dataset = val_dataset.cast_column("audio", Audio(sampling_rate=TARGET_SR))
    print("Casted audio column to Audio(feature) for lazy loading")
except Exception as e:
    print(f"Could not cast to Audio feature (falling back to string). Reason: {e}")
    train_dataset = train_dataset.cast_column("audio", Value("string"))
    val_dataset = val_dataset.cast_column("audio", Value("string"))

# Directory to save datasets and manifests
DATASET_DIR = os.path.join(OUTPUT_DIR, "hf_datasets")
os.makedirs(DATASET_DIR, exist_ok=True)
train_path = os.path.join(DATASET_DIR, "train")
val_path = os.path.join(DATASET_DIR, "validation")
# Save datasets to disk (HF arrow format)
print(f"Saving train dataset to: {train_path}")
train_dataset.save_to_disk(train_path)
print(f"Saving validation dataset to: {val_path}")
val_dataset.save_to_disk(val_path)

# Also write simple CSV manifests (audio path, text, chapter, duration) for convenience
train_manifest = os.path.join(DATASET_DIR, "train_manifest.csv")
val_manifest = os.path.join(DATASET_DIR, "val_manifest.csv")
def write_manifest(list_data, csv_path):
    with open(csv_path, "w", newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["audio","text","chapter","duration"])
        for item in list_data:
            writer.writerow([item.get('audio'), item.get('text'), item.get('chapter'), item.get('duration')])
write_manifest(train_data, train_manifest)
write_manifest(val_data, val_manifest)
print(f"✅ Datasets saved to {DATASET_DIR}")
print(f"   Train samples: {len(train_dataset)}")
print(f"   Val samples: {len(val_dataset)}")
print(f"   Manifests: {train_manifest}, {val_manifest}")

Dataset split:
  Training: 7 chapters
  Validation: 2 chapters
Casted audio column to Audio(feature) for lazy loading
Saving train dataset to: /fs/scratch/<allocation>/output/hf_datasets/train


Saving the dataset (0/1 shards):   0%|          | 0/7 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 7/7 [00:00<00:00, 25.29 examples/s]


Saving validation dataset to: /fs/scratch/<allocation>/output/hf_datasets/validation


Saving the dataset (1/1 shards): 100%|██████████| 2/2 [00:00<00:00, 20.49 examples/s]

✅ Datasets saved to /fs/scratch/<allocation>/output/hf_datasets
   Train samples: 7
   Val samples: 2
   Manifests: /fs/scratch/<allocation>/output/hf_datasets/train_manifest.csv, /fs/scratch/<allocation>/output/hf_datasets/val_manifest.csv


## Processor, Preprocessing and Data Collator
Now we instantiate a processor (feature extractor + tokenizer) used for preprocessing. We intentionally only load the processor (not the heavy model weights) so feature extraction and tokenization can run here. This allows mapping the datasets to `input_values` and `labels` and saving preprocessed datasets to disk.

In [12]:
from transformers import AutoProcessor
# Use a stable/smaller model id for the processor to reduce download size; change if you want the large model's tokenizer/feature-extractor.
processor_model_id = "facebook/wav2vec2-base-960h"
print(f"Loading processor from: {processor_model_id}")
processor = AutoProcessor.from_pretrained(processor_model_id)
print("✅ Processor loaded (feature extractor + tokenizer)")
expected_sr = processor.feature_extractor.sampling_rate
if expected_sr != TARGET_SR:
    print(f"⚠️ Processor expects {expected_sr} Hz, dataset audio is {TARGET_SR} Hz. prepare_dataset will resample if needed.")

Loading processor from: facebook/wav2vec2-base-960h
✅ Processor loaded (feature extractor + tokenizer)
✅ Processor loaded (feature extractor + tokenizer)


In [14]:
import librosa

def _resolve_audio_path(audio_source):
    """Resolve audio path from a datasets Audio feature or a plain string.
    Try several candidate locations (absolute, OUTPUT_DIR, DATASET_DIR) and return the
    first existing path. Returns None if not found.
    """
    # If audio_source is a dict (datasets.Audio feature) it may contain 'path' or 'array'.
    if isinstance(audio_source, dict):
        if 'array' in audio_source and audio_source['array'] is not None:
            return None  # caller will handle in-memory array case
        audio_path = audio_source.get('path')
    else:
        audio_path = audio_source

    if not audio_path:
        return None

    # If absolute and exists, return it
    if os.path.isabs(audio_path) and os.path.exists(audio_path):
        return audio_path

    # Try a few sensible locations where the WAVs might live
    candidates = [
        audio_path,
        os.path.join(OUTPUT_DIR, os.path.basename(audio_path)),
        os.path.join(DATASET_DIR, os.path.basename(audio_path)) if 'DATASET_DIR' in globals() else None,
        os.path.join(PARENT_DIR, audio_path) if 'PARENT_DIR' in globals() else None,
        os.path.join(PARENT_DIR, 'output', os.path.basename(audio_path)) if 'PARENT_DIR' in globals() else None,
    ]
    for c in candidates:
        if not c:
            continue
        if os.path.exists(c):
            return c
    return None

def prepare_dataset(batch):
    audio_source = batch['audio']

    # If the Audio feature already contains the audio array (in-memory), use it directly
    if isinstance(audio_source, dict) and 'array' in audio_source and audio_source['array'] is not None:
        speech = np.array(audio_source['array'], dtype=np.float32)
        sr = audio_source.get('sampling_rate', TARGET_SR)
    else:
        # Resolve path candidates
        audio_path = audio_source.get('path') if isinstance(audio_source, dict) else audio_source
        resolved = _resolve_audio_path(audio_source)
        if resolved is None:
            # Provide detailed error to help debugging
            tried = [
                str(audio_path),
                os.path.join(OUTPUT_DIR, os.path.basename(str(audio_path))) if 'OUTPUT_DIR' in globals() else None,
                os.path.join(DATASET_DIR, os.path.basename(str(audio_path))) if 'DATASET_DIR' in globals() else None,
            ]
            raise FileNotFoundError(f"Could not find audio file. Tried: {', '.join([t for t in tried if t])}")
        speech, sr = sf.read(resolved)

    # Resample if necessary
    if sr != TARGET_SR:
        speech = librosa.resample(np.asarray(speech, dtype=np.float32), orig_sr=sr, target_sr=TARGET_SR)
        sr = TARGET_SR

    inputs = processor.feature_extractor(speech, sampling_rate=sr, return_attention_mask=False)
    batch['input_values'] = inputs.input_values[0]
    batch['input_length'] = len(batch['input_values'])
    batch['labels'] = processor.tokenizer(batch['text']).input_ids
    return batch

print('Preprocessing datasets (this may take a while) ...')
# Map only if datasets do not already have input_values (safe to re-run)
from datasets import load_from_disk
try:
    # Load the saved raw datasets from disk (created above)
    raw_train_path = os.path.join(DATASET_DIR, 'train')
    raw_val_path = os.path.join(DATASET_DIR, 'validation')
    if os.path.exists(raw_train_path):
        train_dataset = load_from_disk(raw_train_path)
    if os.path.exists(raw_val_path):
        val_dataset = load_from_disk(raw_val_path)
except Exception as e:
    print('Could not load raw datasets from disk (will use in-memory variables if present).', e)

# Only map if 'input_values' not present
if 'input_values' not in train_dataset.column_names:
    train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names, num_proc=1)
if 'input_values' not in val_dataset.column_names:
    val_dataset = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names, num_proc=1)
print('✅ Datasets preprocessed (input_values, labels added)')

# Save preprocessed datasets as well
processed_train_path = os.path.join(DATASET_DIR, 'processed_train')
processed_val_path = os.path.join(DATASET_DIR, 'processed_validation')
train_dataset.save_to_disk(processed_train_path)
val_dataset.save_to_disk(processed_val_path)
print(f"✅ Preprocessed datasets saved to: {processed_train_path}, {processed_val_path}")

Preprocessing datasets (this may take a while) ...


Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Map: 100%|██████████| 2/2 [00:00<00:00,  2.21 examples/s]


✅ Datasets preprocessed (input_values, labels added)


Saving the dataset (1/1 shards): 100%|██████████| 2/2 [00:00<00:00, 10.12 examples/s]

✅ Preprocessed datasets saved to: /fs/scratch/<allocation>/output/hf_datasets/processed_train, /fs/scratch/<allocation>/output/hf_datasets/processed_validation


In [15]:
# Data collator definition (same as used for CTC training)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_values': f['input_values']} for f in features]
        label_features = [{'input_ids': f['labels']} for f in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors='pt')
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(label_features, padding=self.padding, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch['labels'] = labels
        return batch
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)
print('✅ Data collator created')

✅ Data collator created


In [16]:
# Save processor to dataset directory so the model notebook can load the same tokenizer/feature-extractor
processor_save_dir = os.path.join(DATASET_DIR, 'processor')
processor.save_pretrained(processor_save_dir)
# Save a small metadata file describing locations for convenience
meta = {
    'DATASET_DIR': DATASET_DIR,
    'raw_train': os.path.join(DATASET_DIR, 'train'),
    'raw_val': os.path.join(DATASET_DIR, 'validation'),
    'processed_train': os.path.join(DATASET_DIR, 'processed_train'),
    'processed_val': os.path.join(DATASET_DIR, 'processed_validation'),
    'processor_dir': processor_save_dir,
    'train_manifest': train_manifest,
    'val_manifest': val_manifest
}
with open(os.path.join(DATASET_DIR, 'dataset_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)
print(f"✅ Processor and metadata saved to {DATASET_DIR}")

✅ Processor and metadata saved to /fs/scratch/<allocation>/output/hf_datasets
